### Project 2 — CFPB Triage Latency Prediction

### Prediction-Time Leakage Audit + Target & Operational EDA

### Objective

- Determine which information is legitimately available when a complaint
is received and understand the operational behavior of the derived
`triage_delay_days` target.

### Prediction question

> At the time a complaint is received, how many days are expected to
> pass before it is sent to the company?

### Prediction timestamp:- `Date received`

### Target:- `triage_delay_days` = `Date sent to company - Date received`

### Core principle

- A feature is valid only if its value is available at or before the
prediction timestamp.

- A column can be statistically predictive and still be invalid for
prediction if it contains information generated after the prediction
event.
-----------
1. What can the model legitimately know at Date received?
2. Which columns are definitely forbidden?
3. Which columns require operational clarification?
4. Is triage delay heavily skewed?
5. How concentrated are delays around immediate processing?
6. How large is the long-delay tail?
7. Does latency differ by Product / Company / time?
8. Are long delays genuine observations or suspicious artifacts?
9. Should we model raw delay or investigate transformation later?
10. What information should flow into the split/design phase?

In [1]:
## Imports & Paths
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent

DATA_PATH = PROJECT_ROOT / "Data" / "raw" / "complaints-cfpb-raw.csv"
REPORT_DIR = PROJECT_ROOT / "Reports" / "project2"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)

Project root: d:\GURU_PROJECTS\ML-FINTECH&BANK
Data path: d:\GURU_PROJECTS\ML-FINTECH&BANK\Data\raw\complaints-cfpb-raw.csv


In [2]:
## Load the Data
df = pd.read_csv(DATA_PATH)

df["Date received"] = pd.to_datetime(
    df["Date received"],
    errors="coerce",
    utc=True
)

df["Date sent to company"] = pd.to_datetime(
    df["Date sent to company"],
    errors="coerce",
    utc=True
)

df["triage_delay_days"] = (
    df["Date sent to company"] - df["Date received"]
).dt.total_seconds() / (24 * 60 * 60)

print("Shape:", df.shape)

Shape: (81946, 17)


# 1. Prediction-Time Leakage Audit

## Prediction event

- The prediction is made when the complaint is received.

Therefore:- `T_pred = Date received`

### Allowed information

- A feature is potentially valid if it is known at or before
`Date received`.

### Forbidden information

- Any information generated after complaint receipt is forbidden.

This includes variables describing downstream processing,
company response, final outcomes, or timestamps occurring after intake.

### Important distinction

There are three categories:

1. SAFE
   Clearly available at intake.

2. FORBIDDEN
   Clearly generated after intake or derived from future information.

3. REQUIRES OPERATIONAL VALIDATION
   The dataset alone cannot prove when the field became available.

In [3]:
## Candidate features classification
feature_audit = pd.DataFrame([
    {
        "column": "Date received",
        "status": "SAFE",
        "reason": "Defines the prediction timestamp."
    },
    {
        "column": "Product",
        "status": "VERIFY",
        "reason": "May be assigned during intake; verify operational availability."
    },
    {
        "column": "Sub-product",
        "status": "VERIFY",
        "reason": "May be assigned during intake; verify operational availability."
    },
    {
        "column": "Issue",
        "status": "VERIFY",
        "reason": "May be assigned during intake; verify operational availability."
    },
    {
        "column": "Sub-issue",
        "status": "VERIFY",
        "reason": "May be assigned during intake; verify operational availability."
    },
    {
        "column": "Consumer complaint narrative",
        "status": "SAFE",
        "reason": "Complaint narrative is available as part of the complaint."
    },
    {
        "column": "Company",
        "status": "SAFE",
        "reason": "The complaint identifies the company being complained about."
    },
    {
        "column": "State",
        "status": "SAFE",
        "reason": "Provided with complaint intake data."
    },
    {
        "column": "ZIP code",
        "status": "SAFE",
        "reason": "Provided with complaint intake data."
    },
    {
        "column": "Tags",
        "status": "VERIFY",
        "reason": "Need to establish whether tags are assigned at intake or later."
    },
    {
        "column": "Submitted via",
        "status": "SAFE",
        "reason": "Submission channel is known at intake, although zero variance in this extract."
    },
    {
        "column": "Date sent to company",
        "status": "FORBIDDEN",
        "reason": "Occurs after the prediction event and directly constructs the target."
    },
    {
        "column": "Company response to consumer",
        "status": "FORBIDDEN",
        "reason": "Downstream outcome after intake."
    },
    {
        "column": "Timely response?",
        "status": "FORBIDDEN",
        "reason": "Downstream response/SLA outcome."
    },
    {
        "column": "Company public response",
        "status": "FORBIDDEN",
        "reason": "Downstream company response information."
    },
    {
        "column": "Complaint ID",
        "status": "IDENTIFIER",
        "reason": "Traceability identifier, not a predictive feature."
    },
])

display(feature_audit)

,column,status,reason
0,Date received,SAFE,Defines the prediction timestamp.
1,Product,VERIFY,May be assigned during intake; verify operatio...
2,Sub-product,VERIFY,May be assigned during intake; verify operatio...
3,Issue,VERIFY,May be assigned during intake; verify operatio...
4,Sub-issue,VERIFY,May be assigned during intake; verify operatio...
5,Consumer complaint narrative,SAFE,Complaint narrative is available as part of th...
6,Company,SAFE,The complaint identifies the company being com...
7,State,SAFE,Provided with complaint intake data.
8,ZIP code,SAFE,Provided with complaint intake data.
9,Tags,VERIFY,Need to establish whether tags are assigned at...


### Important unresolved layer

`Product`, `Sub-product`, `Issue`, and `Sub-issue` must NOT automatically
be rejected as leakage.

Unlike Project 1, their role here is different.

For Product Classification, Product was the target, so downstream
taxonomy fields such as Issue could leak the target.

For Triage Latency Prediction, these fields may potentially be legitimate
if they are already assigned when the complaint enters the triage process.

- Therefore we need to distinguish:-> "Was the field generated before prediction?"

- from:- > "Is the field downstream of the prediction process?"

This is an operational availability question, not something we should
guess from correlation alone.

In [4]:
## Check Zero-Variance/ Unusable Columns
zero_variance = []

for col in df.columns:
    if df[col].nunique(dropna=False) <= 1:
        zero_variance.append(col)

print("Zero-variance columns:")
print(zero_variance)

Zero-variance columns:
['Submitted via']


In [5]:
## Missingness Audit
missing_audit = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
    "unique_values": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)

display(missing_audit)

,missing_count,missing_pct,unique_values
Tags,69230,84.482464,3
Company public response,52692,64.300881,10
Sub-issue,10166,12.405731,205
State,603,0.735850,58
ZIP code,350,0.427111,6780
Date received,0,0.000000,80912
Sub-product,0,0.000000,54
Issue,0,0.000000,86
Product,0,0.000000,11
Company,0,0.000000,1995


#### 2. Target Distribution

The target is expected to be strongly right-skewed.

We therefore examine:

- central tendency
- dispersion
- percentiles
- zero/immediate processing behavior
- long-delay tail
- skewness
- concentration of operational delays

### Important principle

Do not decide yet whether to:

- remove outliers
- winsorize
- clip
- log-transform
- cap the target

Those are modeling decisions.

In [6]:
### Target Statistics
target = df["triage_delay_days"]

target_stats = target.describe(
    percentiles=[
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.975,
        0.99,
        0.995
    ]
)

display(target_stats)

count    81946.000000
mean         1.744687
std          8.058747
min          0.000208
1%           0.000278
5%           0.001678
10%          0.002558
25%          0.005116
50%          0.009907
75%          0.019282
90%          0.041053
95%         10.914462
97.5%       33.101053
99%         43.810261
99.5%       49.281240
max        146.791586
Name: triage_delay_days, dtype: float64

In [ ]:
## Skewness and Concentration
skewness = target.skew()

concentration = pd.Series({
    "mean_days": target.mean(),
    "median_days": target.median(),
    "mean_median_ratio": target.mean() / target.median(),
    "p75_days": target.quantile(0.75),
    "p90_days": target.quantile(0.90),
    "p95_days": target.quantile(0.95),
    "p99_days": target.quantile(0.99),
    "max_days": target.max(),
    "skewness": skewness,
})

display(concentration.to_frame("value"))

# The mean is being pulled upward by a relatively small number of long-delay complaints.
# This is already a warning that MAE should eventually be our primary regression metric, rather than relying on RMSE/R² alone.

,value
mean_days,1.744687
median_days,0.009907
mean_median_ratio,176.099298
p75_days,0.019282
p90_days,0.041053
p95_days,10.914462
p99_days,43.810261
max_days,146.791586
skewness,6.030795


In [8]:
### Operational Delay Bonds
bins = [
    -np.inf,
    0.01,
    0.05,
    0.25,
    1,
    3,
    7,
    14,
    30,
    60,
    90,
    np.inf
]

labels = [
    "<= 0.01d",
    "0.01–0.05d",
    "0.05–0.25d",
    "0.25–1d",
    "1–3d",
    "3–7d",
    "7–14d",
    "14–30d",
    "30–60d",
    "60–90d",
    ">90d"
]

df["delay_band"] = pd.cut(
    target,
    bins=bins,
    labels=labels,
    include_lowest=True
)

delay_band_summary = (
    df["delay_band"]
    .value_counts(sort=False)
    .to_frame("count")
)

delay_band_summary["percentage"] = (
    delay_band_summary["count"] / len(df) * 100
)

display(delay_band_summary)

,count,percentage
delay_band,,
<= 0.01d,41340,50.447856
0.01–0.05d,33559,40.952579
0.05–0.25d,1600,1.952505
0.25–1d,217,0.264809
1–3d,145,0.176946
3–7d,403,0.491787
7–14d,997,1.216655
14–30d,1447,1.765797
30–60d,2113,2.578527


In [9]:
## Immediate vs Delayed Processing
thresholds = [0.01, 0.05, 0.25, 1, 3, 7, 14, 30]

threshold_summary = []

for threshold in thresholds:
    count = (target <= threshold).sum()

    threshold_summary.append({
        "threshold_days": threshold,
        "count_within_threshold": int(count),
        "percentage": count / len(target) * 100
    })

threshold_summary = pd.DataFrame(threshold_summary)

display(threshold_summary)

,threshold_days,count_within_threshold,percentage
0,0.01,41340,50.447856
1,0.05,74899,91.400434
2,0.25,76499,93.352940
3,1.00,76716,93.617748
4,3.00,76861,93.794694
5,7.00,77264,94.286481
6,14.00,78261,95.503136
7,30.00,79708,97.268933


In [10]:
## Target by product
product_latency = (
    df.groupby("Product", dropna=False)["triage_delay_days"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        p75=lambda x: x.quantile(0.75),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        max="max"
    )
    .sort_values("median", ascending=False)
)

display(product_latency)

,count,mean,median,p75,p90,p95,max
Product,,,,,,,
Mortgage,5336,1.752029,0.014873,0.026418,0.051395,11.312115,120.233322
Vehicle loan or lease,3993,1.841161,0.012928,0.022917,0.047109,12.948333,79.220174
"Payday loan, title loan, personal loan, or advance loan",2898,2.707728,0.012494,0.022454,0.074834,25.977100,118.248044
Credit card,13820,1.705626,0.012072,0.021863,0.045514,11.143104,118.030035
"Money transfer, virtual currency, or money service",6545,1.848760,0.011736,0.021273,0.041898,11.543222,116.847153
Student loan,2687,1.838284,0.011678,0.021586,0.690331,11.063012,114.294664
Prepaid card,912,2.359345,0.011591,0.020946,0.068825,21.063010,79.015162
Checking or savings account,14673,0.958204,0.011192,0.019653,0.035044,0.074873,137.814595
Debt or credit management,719,3.516182,0.009676,0.021285,13.335894,32.229285,62.712801


In [11]:
## Target By company
company_latency = (
    df.groupby("Company", dropna=False)["triage_delay_days"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        p95=lambda x: x.quantile(0.95)
    )
    .sort_values("count", ascending=False)
)

display(company_latency.head(20))

,count,mean,median,p95
Company,,,,
CAPITAL ONE FINANCIAL CORPORATION,3233,0.849592,0.010521,0.066028
WELLS FARGO & COMPANY,3189,0.789081,0.011667,0.057005
"CITIBANK, N.A.",2965,1.409860,0.011991,4.858782
"BANK OF AMERICA, NATIONAL ASSOCIATION",2944,0.806286,0.011771,0.064993
JPMORGAN CHASE & CO.,2933,0.791924,0.011725,0.058954
"Block, Inc.",2822,1.217576,0.010469,6.433574
"TRANSUNION INTERMEDIATE HOLDINGS, INC.",2396,0.543966,0.003067,0.022347
"Paypal Holdings, Inc",2107,0.925774,0.010671,0.060502
CL Holdings LLC,2075,0.783318,0.004317,0.035766


### Company interpretation warning

Company-level latency differences are descriptive.

They do NOT establish that a company causes longer or shorter triage
processing.

Observed differences may reflect:

- complaint mix
- volume
- issue complexity
- operational routing
- reporting behavior
- time period
- other unobserved factors

Therefore this analysis is for prediction and operational understanding,
not causal attribution.

In [12]:
## Temporal  Operational Analysis
df["received_date"] = df["Date received"].dt.date
df["received_month"] = df["Date received"].dt.to_period("M").astype(str)
df["received_dayofweek"] = df["Date received"].dt.day_name()
df["received_hour"] = df["Date received"].dt.hour

monthly_latency = (
    df.groupby("received_month")["triage_delay_days"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95)
    )
)

display(monthly_latency)

C:\Users\MY PC\AppData\Local\Temp\ipykernel_3440\1688712747.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["received_month"] = df["Date received"].dt.to_period("M").astype(str)


,count,mean,median,p90,p95
received_month,,,,,
2026-03,21630,2.418575,0.009398,0.058134,21.671564
2026-04,21048,1.665547,0.010069,0.037376,6.914550
2026-05,17385,2.523876,0.010150,0.054220,22.946891
2026-06,16721,0.685929,0.010023,0.034688,0.084803
2026-07,5161,0.049010,0.010058,0.033438,0.054896
2026-08,1,0.003090,0.003090,0.003090,0.003090


In [13]:
## Day-Of-Week
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_latency = (
    df.groupby("received_dayofweek")["triage_delay_days"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        p95=lambda x: x.quantile(0.95)
    )
    .reindex(weekday_order)
)

display(weekday_latency)

,count,mean,median,p95
received_dayofweek,,,,
Monday,15687,1.516332,0.009861,8.163022
Tuesday,17646,1.471264,0.009630,6.948423
Wednesday,17725,1.425679,0.009699,5.769428
Thursday,15786,1.368905,0.009595,4.795437
Friday,6747,3.080722,0.010671,27.496019
Saturday,1895,7.648476,0.015058,42.207199
Sunday,6460,1.712439,0.010666,10.729166


In [14]:
## Hourly Operational Pattern
hourly_latency = (
    df.groupby("received_hour")["triage_delay_days"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        p95=lambda x: x.quantile(0.95)
    )
)

display(hourly_latency)

,count,mean,median,p95
received_hour,,,,
0,3052,2.351460,0.011042,18.562492
1,2786,2.240727,0.011094,17.217934
2,2550,2.113160,0.010660,16.662299
3,2287,2.034060,0.011458,15.366564
4,2104,1.659491,0.010584,8.377698
5,1706,1.520565,0.009786,7.103076
6,1372,0.960950,0.009063,0.061135
7,1069,1.059332,0.008773,0.103294
8,977,1.039973,0.009097,0.068646


In [15]:
## Long-Delay Investigation
long_delay_thresholds = [1, 3, 7, 14, 30, 60, 90]

long_delay_summary = []

for threshold in long_delay_thresholds:
    mask = target > threshold

    long_delay_summary.append({
        "delay_greater_than_days": threshold,
        "count": int(mask.sum()),
        "percentage": mask.mean() * 100
    })

long_delay_summary = pd.DataFrame(long_delay_summary)

display(long_delay_summary)

,delay_greater_than_days,count,percentage
0,1,5230,6.382252
1,3,5085,6.205306
2,7,4682,5.713519
3,14,3685,4.496864
4,30,2238,2.731067
5,60,125,0.152539
6,90,64,0.078100


In [16]:
## Inspect Etreme Cases
extreme_cases = (
    df[
        [
            "Complaint ID",
            "Date received",
            "Date sent to company",
            "Product",
            "Sub-product",
            "Issue",
            "Company",
            "State",
            "triage_delay_days"
        ]
    ]
    .sort_values("triage_delay_days", ascending=False)
)

display(extreme_cases.head(30))

,Complaint ID,Date received,Date sent to company,Product,Sub-product,Issue,Company,State,triage_delay_days
14361,20379834,2026-03-18 17:19:32+00:00,2026-08-12 12:19:25+00:00,Debt collection,Credit card debt,Attempts to collect debt not owed,"Hayt Hayt & Landau, P.L. (FL)",GA,146.791586
21102,19987390,2026-03-05 02:00:50+00:00,2026-07-22 18:22:58+00:00,Debt collection,I do not know,Attempts to collect debt not owed,"CITIBANK, N.A.",TX,139.682037
9303,20273964,2026-03-14 22:13:44+00:00,2026-07-30 20:34:42+00:00,Debt collection,Telecommunications debt,Written notification about debt,"MRS BPO, LLC",NC,137.931227
14222,20737733,2026-03-28 00:54:21+00:00,2026-08-12 20:27:22+00:00,Checking or savings account,Checking account,Problem with a lender or other company chargin...,Albert Corporation,CA,137.814595
78387,20420820,2026-03-19 18:53:27+00:00,2026-07-30 20:24:03+00:00,Debt collection,Rental debt,Written notification about debt,"FAIR COLLECTIONS & OUTSOURCING, INC.",PA,133.062917
23714,20021840,2026-03-05 23:56:01+00:00,2026-07-14 15:17:53+00:00,Debt collection,Telecommunications debt,Attempts to collect debt not owed,"I.C. System, Inc.",FL,130.640185
21825,20091651,2026-03-09 14:39:30+00:00,2026-07-07 20:15:29+00:00,Mortgage,Conventional home mortgage,Closing on a mortgage,"JHL, LLC",FL,120.233322
81553,19949603,2026-03-04 00:29:05+00:00,2026-07-01 13:04:09+00:00,Debt collection,Mortgage debt,False statements or representation,Concord Servicing Corporation,NC,119.524352
45895,20418771,2026-03-19 17:51:22+00:00,2026-07-16 17:52:48+00:00,Mortgage,FHA mortgage,Applying for a mortgage or refinancing an exis...,Liberty 1 Lending Inc.,IN,119.000995
74095,19980225,2026-03-04 21:37:32+00:00,2026-07-01 13:13:30+00:00,Debt collection,Other debt,Attempts to collect debt not owed,Self Financial Inc.,FL,118.649977


In [17]:
## Target Uniquess
target_frequency = (
    target
    .round(6)
    .value_counts()
    .head(20)
    .to_frame("count")
)

display(target_frequency)

,count
triage_delay_days,
0.000289,340
0.000278,336
0.000301,320
0.000266,247
0.000312,240
0.000255,191
0.000324,176
0.000336,128
0.000243,85


In [18]:
### Narrative Length vs Latency
df["narrative_word_count"] = (
    df["Consumer complaint narrative"]
    .fillna("")
    .str.split()
    .str.len()
)

narrative_latency = (
    df.groupby(
        pd.qcut(
            df["narrative_word_count"],
            q=10,
            duplicates="drop"
        )
    )["triage_delay_days"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        p95=lambda x: x.quantile(0.95)
    )
)

display(narrative_latency)

,count,mean,median,p95
narrative_word_count,,,,
"(4.999, 51.0]",8209,2.060408,0.005972,16.185269
"(51.0, 75.0]",8267,1.326816,0.004954,0.872527
"(75.0, 111.0]",8193,1.814615,0.008646,12.791894
"(111.0, 140.0]",8192,2.037021,0.008368,14.878991
"(140.0, 175.0]",8233,1.638869,0.010012,9.879775
"(175.0, 215.0]",8084,1.745787,0.011441,11.204141
"(215.0, 262.0]",8274,1.757168,0.012581,11.725976
"(262.0, 327.0]",8143,1.803827,0.013160,11.497748
"(327.0, 447.0]",8169,1.638157,0.013808,8.828384


In [19]:
## Target Transformation Investigation
target_shape = pd.DataFrame({
    "raw_target": target,
    "log1p_target": np.log1p(target)
})

display(
    target_shape.describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

,raw_target,log1p_target
count,81946.000000,81946.000000
mean,1.744687,0.210913
std,8.058747,0.777840
min,0.000208,0.000208
1%,0.000278,0.000278
25%,0.005116,0.005103
50%,0.009907,0.009859
75%,0.019282,0.019099
90%,0.041053,0.040233
95%,10.914462,2.477753


### 3. Decision Framework

### Current observations

The target is:

- non-negative
- continuous
- highly right-skewed
- concentrated near very small delays
- associated with a substantial long-delay tail

### Therefore

We should NOT assume ordinary raw-scale regression is automatically
the best modeling formulation.

However, we also should NOT transform the target simply because it
looks skewed.

### Candidate formulations for later experimentation

1. Raw-target regression

   Predict:
   `triage_delay_days`

2. Log-target regression

   Predict:
   `log1p(triage_delay_days)`

3. Potential operational threshold formulation

   Later, if the business defines an SLA threshold, consider a separate
   classification problem such as:

   `delay > SLA_threshold`

These are different business questions and should not be mixed.

### Current decision

Keep the original target as the canonical target.

Investigate `log1p` only as a modeling experiment later.

Do not alter or delete extreme observations at this stage.

In [20]:
## Leakage Decision Summary
leakage_summary = pd.DataFrame([
    ["Date received", "SAFE", "Prediction timestamp"],
    ["Product", "VERIFY", "Operational availability must be confirmed"],
    ["Sub-product", "VERIFY", "Operational availability must be confirmed"],
    ["Issue", "VERIFY", "Operational availability must be confirmed"],
    ["Sub-issue", "VERIFY", "Operational availability must be confirmed"],
    ["Consumer complaint narrative", "SAFE", "Available complaint content"],
    ["Company", "SAFE", "Known complaint entity"],
    ["State", "SAFE", "Available intake metadata"],
    ["ZIP code", "SAFE", "Available intake metadata"],
    ["Tags", "VERIFY", "Need intake-time availability confirmation"],
    ["Submitted via", "DROP", "Zero variance in current extract"],
    ["Date sent to company", "FORBIDDEN", "Future information / target construction"],
    ["Company response to consumer", "FORBIDDEN", "Post-intake outcome"],
    ["Timely response?", "FORBIDDEN", "Post-intake outcome"],
    ["Company public response", "FORBIDDEN", "Post-intake response"],
    ["Complaint ID", "DROP", "Identifier only"],
], columns=["column", "status", "reason"])

display(leakage_summary)

,column,status,reason
0,Date received,SAFE,Prediction timestamp
1,Product,VERIFY,Operational availability must be confirmed
2,Sub-product,VERIFY,Operational availability must be confirmed
3,Issue,VERIFY,Operational availability must be confirmed
4,Sub-issue,VERIFY,Operational availability must be confirmed
5,Consumer complaint narrative,SAFE,Available complaint content
6,Company,SAFE,Known complaint entity
7,State,SAFE,Available intake metadata
8,ZIP code,SAFE,Available intake metadata
9,Tags,VERIFY,Need intake-time availability confirmation


#### Final Analysis Insights

#### 1. Prediction boundary

The prediction is made at `Date received`.

Therefore all future information must be excluded from the feature set.

#### 2. Target

`triage_delay_days` is a valid derived operational latency target:

`Date sent to company - Date received`

#### 3. Target shape

The target is strongly right-skewed.

The median delay is extremely small relative to the mean, while the
upper tail contains substantially longer delays.

#### 4. Operational interpretation

The dataset appears to contain two very different operational regimes:

- rapid/immediate processing
- substantially delayed processing

This makes the tail operationally important rather than automatically
treating it as statistical noise.

#### 5. Outlier policy

Extreme delays are retained.

They will only be reconsidered if later investigation demonstrates
data-quality problems.

#### 6. Feature availability

Clearly safe candidates include complaint intake information such as
narrative, company, state and ZIP.

Product taxonomy fields and tags require operational availability
validation rather than automatic inclusion or exclusion.

#### 7. Forbidden information

Date sent to company, response fields and downstream SLA/outcome fields
are forbidden predictors.

#### 8. Modeling implication

- MAE should eventually be the primary regression metric because it measures
typical prediction error directly in days and is less dominated by the
extreme tail than RMSE.

- RMSE remains useful as a secondary metric because large delay
underprediction may have operational importance.

#### 9. Transformation

The canonical target remains the raw `triage_delay_days`.

`log1p` transformation is a candidate modeling experiment, not a
preprocessing decision yet.

#### 10. Next phase:- The next stage is:- `Split Design`

- The split must respect the temporal nature of operational prediction
rather than blindly copying the Product Classification split.